# **Sentimen Analisis Review Wondr BNI - Multinomial Naive Bayes (Standardized)**

## **1. Import Library**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import joblib

print("Library berhasil diimpor!")

## **2. Import Dataset**

In [ ]:
df = pd.read_csv("reviews_wondr_bni.csv")
print(f"Jumlah data: {df.shape}")
df.head()

## **3. Preprocessing (6 Langkah Standar)**

In [ ]:
# Load Stopwords
stopword_df = pd.read_csv('stopwordbahasa.csv', header=None, names=['stopwords'])
stopwords_id = set(stopword_df['stopwords'].values)

# Inisialisasi Stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Kamus Normalisasi (Slang to Formal Indonesian)
norm_dict = {
    "yg": "yang", "gk": "tidak", "tdk": "tidak", "bgt": "banget", "gpp": "tidak apa-apa",
    "kl": "kalau", "klo": "kalau", "udah": "sudah", "sdh": "sudah", "aja": "saja",
    "ga": "tidak", "gak": "tidak", "kmrn": "kemarin", "skrg": "sekarang", "tp": "tapi",
    "dgn": "dengan", "dlm": "dalam", "utk": "untuk", "bisaa": "bisa", "mantap": "bagus",
    "oke": "baik", "ok": "baik", "sip": "baik", "kalo": "kalau", "biar": "supaya",
    "krn": "karena", "karna": "karena", "bngt": "banget", "udh": "sudah", "sy": "saya"
}

def preprocess_steps(text):
    # 1. Cleaning
    text = re.sub(r'@[A-Za-z0-9_]+', '', text) # Mentions
    text = re.sub(r'#[A-Za-z0-9_]+', '', text) # Hashtags
    text = re.sub(r'http\S+|www.\S+', '', text) # Links
    text = re.sub(r'\d+', '', text) # Numbers
    text = text.translate(str.maketrans('', '', string.punctuation)) # Punctuation
    text = re.sub(r'\s+', ' ', text).strip() # Whitespace
    
    # 2. Case Folding
    text = text.lower()
    
    # 3. Normalisasi
    words = text.split()
    words = [norm_dict.get(word, word) for word in words]
    
    # 4. Tokenizing
    
    # 5. Stopword Removal
    words = [word for word in words if word not in stopwords_id]
    
    # 6. Stemming
    words = [stemmer.stem(word) for word in words]
    
    return " ".join(words)

# Labeling Sentimen
def label_sentiment(score):
    if score >= 4: return 'positif'
    elif score == 3: return 'netral'
    else: return 'negatif'

# Terapkan Preprocessing
df_clean = df[['content', 'score']].copy()
df_clean.dropna(inplace=True)

# Sampling 10,000 data
df_sampled = df_clean.sample(n=10000, random_state=42).reset_index(drop=True)

print("Memulai preprocessing... (mungkin memakan waktu beberapa menit)")
df_sampled['text_final'] = df_sampled['content'].astype(str).apply(preprocess_steps)
df_sampled['sentiment'] = df_sampled['score'].apply(label_sentiment)
print("Preprocessing selesai!")
df_sampled.head()

## **4. TF-IDF Vectorization**

In [ ]:
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df_sampled['text_final'])
y = df_sampled['sentiment']

print(f"Fitur TF-IDF berhasil diekstrak: {X.shape}")

## **5. Multinomial Naive Bayes Training**

In [ ]:
# Split Data 75:25
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Latih Model
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

print("Model Multinomial Naive Bayes berhasil dilatih!")

## **6. Evaluasi Sistem (Confusion Matrix & Metrics)**

In [ ]:
y_pred = nb_model.predict(X_test)

# Hitung Metrik
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

print("=== HASIL EVALUASI NAIVE BAYES ===")
print(f"Akurasi  : {accuracy:.4f}")
print(f"Presisi  : {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")

print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Visualisasi Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=nb_model.classes_, yticklabels=nb_model.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Naive Bayes')
plt.savefig('confusion_matrix.png') # Simpan untuk aplikasi web
plt.show()

## **7. Simpan Model**

In [ ]:
joblib.dump(nb_model, 'model_nb_tfidf.h5')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
print("Model dan Vectorizer berhasil disimpan!")